In [1]:
# Run the for loop in each function to return a dictionary
# Make a seperate function that turns each dictionary into a database 

In [2]:
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd
import requests
import sqlite3
import os
import yfinance as yf

In [3]:
load_dotenv()

True

In [4]:
fmg_api_key = os.getenv("FMG_API_KEY")
tiingo_key = os.getenv("TIINGO_TOKEN")

In [5]:
symbols_string = "AAPL, TSLA, AMZN, MSFT, NVDA, GOOGL, META, NFLX, JPM, V, BAC, PYPL, DIS, T, PFE, COST, INTC, KO, TGT, NKE, SPY, BA, BABA, XOM, WMT, GE, CSCO, VZ, JNJ, CVX, PLTR, SQ, SHOP, SBUX, SOFI, HOOD, RBLX, SNAP, AMD, UBER, FDX, ABBV, ETSY, MRNA, LMT, GM, F, LCID, CCL, DAL, UAL, AAL, TSM, SONY, ET, MRO, COIN, RIVN, RIOT, CPRX, VWO, SPYG, NOK, ROKU, VIAC, ATVI, BIDU, DOCU, ZM, PINS, TLRY, WBA, MGM, NIO, C, GS, WFC, ADBE, PEP, UNH, CARR, HCA, TWTR, BILI, SIRI, FUBO, RKT"
symbols_split = symbols_string.split(',')
symbols = [s.strip() for s in symbols_split]

### Get Financial Statements

In [42]:
def get_income_statement(ticker, key):
    # This function will return the income statement, balance sheet, and cash flow statement of a company
    income_endpoint = f"https://financialmodelingprep.com/stable/income-statement?symbol={ticker}&apikey={key}"
    income_requests = requests.get(income_endpoint)
    income_data = income_requests.json()
    return income_data

def get_balance_sheet(ticker, key):
    balance_endpoint = f"https://financialmodelingprep.com/stable/balance-sheet-statement?symbol={ticker}&apikey={key}"
    balance_requests = requests.get(balance_endpoint)
    balance_data = balance_requests.json()
    return balance_data

def get_cash_flow_statement(ticker, key):
    cash_endpoint = f"https://financialmodelingprep.com/stable/cash-flow-statement?symbol={ticker}&apikey={key}"
    cash_requests = requests.get(cash_endpoint)
    cash_data = cash_requests.json()
    return cash_data

def get_income_growth(ticker, key):
    income_growth_endpoint = f"https://financialmodelingprep.com/stable/income-statement-growth?symbol={ticker}&apikey={key}"
    income_growth_requests = requests.get(income_growth_endpoint)
    income_growth_data = income_growth_requests.json()
    return income_growth_data

### Price History

In [7]:
def yfinance_historic_price(ticker):
    yfinance_ticker = yf.Ticker(ticker)
    price_history = yfinance_ticker.history('5y')
    price_history = price_history.reset_index()

    return price_history

### Share History

In [8]:
def yfinance_get_shares(ticker):
    yfinance_ticker = yf.Ticker(ticker)
    shares = yfinance_ticker.get_shares_full(start='2021-10-01')
    df_shares = pd.DataFrame(data=shares,  index=None,)
    df_shares.reset_index(inplace=True)
    df_shares = df_shares.rename(columns={'index': 'Date', 0: 'Shares'})
    df_shares['Date'] = pd.to_datetime(df_shares['Date']).dt.date

    return df_shares

In [9]:
def get_fiscal_years(ticker, key):
    income_statement = get_income_statement(ticker=ticker, k=key)

    fiscal_years = []
    for year in income_statement:
        fiscal_year = year['fiscalYear']
        fiscal_years.append(fiscal_year)

    return fiscal_years

In [22]:
fiscal_years = get_fiscal_years(ticker='AAPL', key=fmg_api_key)
fiscal_years_dict = {'Fiscal Year': fiscal_years}

In [10]:
test_stocks = ['AAPL', 'TSLA', 'AMZN']

### P/B ratio function

In [16]:
pb_ratios_list = []
def calc_pb_ratio(ticker, balance):
    # Call api to obtain company balance sheet
    result = []
    for year in balance:
        # Find filing date
        file_date = year['filingDate']
        
        # Find price on filing date
        price_history = yfinance_historic_price(ticker=ticker)
        filing_date_stock_price = price_history[price_history['Date'] == file_date]
        
        # Convert file date to datetime for next step
        file_date = pd.to_datetime(file_date)
        # Find closest date on shares df
        shares_history = yfinance_get_shares(ticker=ticker)
        shares_history['Date'] = pd.to_datetime(shares_history['Date'])
        closest_date = (shares_history['Date'] - file_date).abs().idxmin()
        closest_row = shares_history.loc[closest_date]
        # Calculate Market Cap
        market_cap = closest_row['Shares'] * filing_date_stock_price['Close'].iloc[0]
        market_cap = float(market_cap)

        # Calculate pb ratio
        pb_ratio = round(market_cap / year['totalStockholdersEquity'], 2)
        result.append(pb_ratio)
    
    return result

for stock in test_stocks:
    balance_sheet = get_balance_sheet(ticker=stock, key=fmg_api_key)
    pb_ratio = calc_pb_ratio(ticker=stock, balance=balance_sheet)
    pb_ratios_list.append(pb_ratio)
    
pb_ratios_list


[[54.26, 60.01, 43.88, 48.0, 38.03],
 [19.03, 17.62, 9.71, 12.26, 10.35],
 [5.49, 8.43, 8.84, 7.25, 0.58]]

In [18]:
pb_ratios_dict = dict(zip(test_stocks, pb_ratios_list))
pb_ratios_dict

{'AAPL': [54.26, 60.01, 43.88, 48.0, 38.03],
 'TSLA': [19.03, 17.62, 9.71, 12.26, 10.35],
 'AMZN': [5.49, 8.43, 8.84, 7.25, 0.58]}

In [23]:
pb_ratios_full_dict = fiscal_years_dict | pb_ratios_dict
pb_ratios_full_dict

{'Fiscal Year': ['2025', '2024', '2023', '2022', '2021'],
 'AAPL': [54.26, 60.01, 43.88, 48.0, 38.03],
 'TSLA': [19.03, 17.62, 9.71, 12.26, 10.35],
 'AMZN': [5.49, 8.43, 8.84, 7.25, 0.58]}

In [24]:
df_pb_ratios = pd.DataFrame.from_dict(data=pb_ratios_full_dict)
df_pb_ratios

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,54.26,19.03,5.49
1,2024,60.01,17.62,8.43
2,2023,43.88,9.71,8.84
3,2022,48.00,12.26,7.25
4,2021,38.03,10.35,0.58


### Debt to Equity function

In [25]:
def calc_de_ratio(ticker, balance):
    de_ratios = []
    for year in balance:
        # Step 1: Pull shareholder and total debt from balance sheet
        shareholder_equity = year['totalStockholdersEquity']
        total_debt = year['totalDebt']
        # Step 2: Divide numbers
        de_ratio = total_debt / shareholder_equity
        # Step 3: Append yearly ratios to list
        de_ratios.append(de_ratio)

    return de_ratios

In [26]:
de_ratios_list = []
for stock in test_stocks:
    balance_sheet = get_balance_sheet(ticker=stock, key=fmg_api_key)
    de_ratio = calc_de_ratio(ticker=stock, balance=balance_sheet)
    de_ratios_list.append(de_ratio)

de_ratios_list

[[1.5241072518411023,
  2.090588235294118,
  1.9941750072410132,
  2.6144616356173036,
  2.163924552226977],
 [0.1019759669819935,
  0.18683910962379824,
  0.15284031037455695,
  0.12857909806728704,
  0.2939150021531021],
 [0.37217228418863196,
  0.45774032241144175,
  0.6717572755417957,
  0.9594297569893798,
  0.8419472675322797]]

In [27]:
de_ratios_dict = dict(zip(test_stocks, de_ratios_list))
de_ratios_dict

{'AAPL': [1.5241072518411023,
  2.090588235294118,
  1.9941750072410132,
  2.6144616356173036,
  2.163924552226977],
 'TSLA': [0.1019759669819935,
  0.18683910962379824,
  0.15284031037455695,
  0.12857909806728704,
  0.2939150021531021],
 'AMZN': [0.37217228418863196,
  0.45774032241144175,
  0.6717572755417957,
  0.9594297569893798,
  0.8419472675322797]}

In [28]:
de_ratio_full = fiscal_years_dict | de_ratios_dict
de_ratio_full

{'Fiscal Year': ['2025', '2024', '2023', '2022', '2021'],
 'AAPL': [1.5241072518411023,
  2.090588235294118,
  1.9941750072410132,
  2.6144616356173036,
  2.163924552226977],
 'TSLA': [0.1019759669819935,
  0.18683910962379824,
  0.15284031037455695,
  0.12857909806728704,
  0.2939150021531021],
 'AMZN': [0.37217228418863196,
  0.45774032241144175,
  0.6717572755417957,
  0.9594297569893798,
  0.8419472675322797]}

In [29]:
df_de_ratio = pd.DataFrame.from_dict(de_ratio_full)
df_de_ratio

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,1.524107,0.101976,0.372172
1,2024,2.090588,0.186839,0.457740
2,2023,1.994175,0.152840,0.671757
3,2022,2.614462,0.128579,0.959430
4,2021,2.163925,0.293915,0.841947


### Revenue Growth

In [35]:
def calc_revenue_growth(ticker, income_growth):
    revenue_growth = []
    for year in income_growth:
        growth = year['growthRevenue']
        revenue_growth.append(growth)

    return revenue_growth

[0.0642551178283274,
 0.020219940775141214,
 -0.028004605303199367,
 0.07793787604184606,
 0.33259384733074693]

In [37]:
revenue_growth_list = []
for stock in test_stocks:
    income_growth_statement = get_income_growth(ticker=stock, key=fmg_api_key)
    revenue_growth = calc_revenue_growth(ticker=stock, income_growth=income_growth_statement)
    revenue_growth_list.append(revenue_growth)

In [38]:
revenue_growth_dict = dict(zip(test_stocks, revenue_growth_list))

In [39]:
revenue_growth_full = fiscal_years_dict | revenue_growth_dict

In [40]:
df_revenue_growth = pd.DataFrame.from_dict(data=revenue_growth_full)
df_revenue_growth

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,0.064255,-0.029307,0.123778
1,2024,0.020220,0.009476,0.109909
2,2023,-0.028005,0.187953,0.118296
3,2022,0.077938,0.513517,0.093995
4,2021,0.332594,0.706716,0.216954


### Gross Profit Margins

In [44]:
def calc_gross_profit_margin(ticker, income):
    gross_profit_margin = []
    for year in income:
        total_revenue = year['revenue']
        gross_profit = year['grossProfit']
        gpm = gross_profit / total_revenue
        gross_profit_margin.append(gpm)

    return gross_profit_margin

In [45]:
gpm_list = []

for stock in test_stocks:
    income_statement = get_income_statement(ticker=stock, key=fmg_api_key)
    gross_profit_margin = calc_gross_profit_margin(ticker=stock, income=income_statement)
    gpm_list.append(gross_profit_margin)

[[0.4690516410716045,
  0.4620634981523393,
  0.4413112957720756,
  0.43309630561360085,
  0.4177935962516778],
 [0.18026511436616155,
  0.17862626676220697,
  0.18248891736331416,
  0.25598438535759005,
  0.25279155751258753],
 [0.5028566486824266,
  0.48854393464156787,
  0.46982088955000567,
  0.43805339865326287,
  0.420325144416396]]

In [47]:
gpm_dict = dict(zip(test_stocks, gpm_list))

In [48]:
gpm_full = fiscal_years_dict | gpm_dict

In [49]:
df_gross_profit_margin = pd.DataFrame.from_dict(gpm_full)
df_gross_profit_margin

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,0.469052,0.180265,0.502857
1,2024,0.462063,0.178626,0.488544
2,2023,0.441311,0.182489,0.469821
3,2022,0.433096,0.255984,0.438053
4,2021,0.417794,0.252792,0.420325


In [ ]:
gross_profit_margin = []

for year in income:
    total_revenue = year['revenue']
    gross_profit = year['grossProfit']
    gpm = gross_profit / total_revenue
    gross_profit_margin.append(gpm)

gross_profit_margin

In [9]:

# Debt to equity
de_dict = {}
# Revenue Growth
revenue_growth_dict = {}
# Gross Profit margin
gross_margin_dict = {}
# Altman z score
z_score = {}

In [31]:
pb_keys = ['Fiscal Years']
pb_values = []
keys_to_delete = []

In [28]:
fiscal_years = get_fiscal_years(ticker="AAPL", key=api_key)
pb_values.append(fiscal_years)

In [30]:
for s in symbols:
    # Step 1: Obtain financial statements
    income_statement = get_income_statement(ticker=s,)
    balance_sheet = get_balance_sheet(ticker=s,)
    cash_flow_stat = get_cash_flow_statement(ticker=s,)
    # Step 2: Calculate P/B ratios
    pb_ratios, skipped = calc_pb_ratio(ticker=s, key=api_key), bala
    pb_values.append(pb_ratios)
    keys_to_delete.append(skipped)

$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$SQ: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symb

In [ ]:
pb_ratios

In [ ]:
# PB ratio
pb_ratio_dict = dict(zip(pb_keys